# NVE sweep at constant volume rate

This is the single-axis counterpart to `NVE_Sliding.ipynb`. Only one box length is changed while the other two remain fixed. Because $V=L_xL_yL_z$, making the selected length linear in time makes $dV/dt$ constant. Density is consequently *not* linear in time.

The first branch compresses to `rho_high`; the second expands back to the original density. Each pair is repeated for `number_of_cycles`, and the sweep is repeated for each requested number of steps per leg.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    open_run,
    run_clone_rescale_constant_volume_rate,
)

# --------------------------------------------------
# Configuration
# --------------------------------------------------
database = SQLiteRunDatabase(ProjectPaths().database)
original_run = "20260910171124"
nsteps_per_leg = [2_050, 2_500, 5_000, 10_000, 25_000, 50_000]
number_of_cycles = 3
rho_high = 0.7
resize_axis = "x"  # May be "x", "y", or "z"
density_grid_points = 250
smoothing_window = 1

records = database.query_thermalizations(Run_ID=original_run)
if not records:
    raise ValueError(f"Run {original_run} was not found")
rho_original = float(records[0]["Density_End"])
target_densities = [rho_high, rho_original]
number_of_legs = 2 * number_of_cycles

sweep_runs = {}
for nsteps in nsteps_per_leg:
    print(f"\nRunning {nsteps:,} steps per leg")
    source_run_id = original_run
    run_ids = []

    for leg_index in range(number_of_legs):
        target_density = target_densities[leg_index % 2]
        direction = "compression" if target_density > rho_original else "expansion"
        result = run_clone_rescale_constant_volume_rate(
            source_run_id=source_run_id,
            final_density=target_density,
            nsteps=nsteps,
            axis=resize_axis,
            ensemble="NVE",
            notes=(
                f"Constant-dV/dt NVE triangle; axis={resize_axis}; "
                f"Nsteps={nsteps}; cycle={leg_index // 2 + 1}; "
                f"direction={direction}"
            ),
        )
        source_run_id = result["run_id"]
        run_ids.append(source_run_id)
        print(f"  Leg {leg_index + 1}: {direction}, Run_ID={source_run_id}")

    sweep_runs[nsteps] = run_ids



Running 2,050 steps per leg
  Leg 1: compression, Run_ID=20260914184425
  Leg 2: expansion, Run_ID=20260914184456
  Leg 3: compression, Run_ID=20260914184525
  Leg 4: expansion, Run_ID=20260914184553
  Leg 5: compression, Run_ID=20260914184623
  Leg 6: expansion, Run_ID=20260914184653

Running 2,500 steps per leg
  Leg 1: compression, Run_ID=20260914184722
  Leg 2: expansion, Run_ID=20260914184755
  Leg 3: compression, Run_ID=20260914184830
  Leg 4: expansion, Run_ID=20260914184904
  Leg 5: compression, Run_ID=20260914184938
  Leg 6: expansion, Run_ID=20260914185010

Running 5,000 steps per leg
  Leg 1: compression, Run_ID=20260914185044
  Leg 2: expansion, Run_ID=20260914185144
  Leg 3: compression, Run_ID=20260914185243
  Leg 4: expansion, Run_ID=20260914185342
  Leg 5: compression, Run_ID=20260914185443
  Leg 6: expansion, Run_ID=20260914185545

Running 10,000 steps per leg
  Leg 1: compression, Run_ID=20260914185645
  Leg 2: expansion, Run_ID=20260914190156


In [ ]:
# Average the repeated compression and expansion branches, then plot their residual.
averaged_curves = {}
for nsteps in nsteps_per_leg:
    branches = {"compression": [], "expansion": []}
    for run_id in sweep_runs[nsteps]:
        logs = open_run(run_id).logs_dataframe()
        density = logs["density"].to_numpy(dtype=float)
        pressure = logs["pressure"].to_numpy(dtype=float)
        direction = "compression" if density[-1] > density[0] else "expansion"
        branches[direction].append((density, pressure))

    averaged_curves[nsteps] = {}
    for direction, runs in branches.items():
        if len(runs) != number_of_cycles:
            raise RuntimeError(
                f"Expected {number_of_cycles} {direction} legs for {nsteps:,} steps; "
                f"found {len(runs)}"
            )
        rho_min = max(density.min() for density, _ in runs)
        rho_max = min(density.max() for density, _ in runs)
        grid = np.linspace(rho_min, rho_max, density_grid_points)
        pressure_grid = np.asarray([
            np.interp(grid, density[np.argsort(density)], pressure[np.argsort(density)])
            for density, pressure in runs
        ])
        mean = pressure_grid.mean(axis=0)
        smoothed = (
            pd.Series(mean)
            .rolling(smoothing_window, center=True, min_periods=1)
            .mean()
            .to_numpy()
        )
        averaged_curves[nsteps][direction] = {
            "density": grid,
            "pressure": smoothed,
            "std": pressure_grid.std(axis=0, ddof=1),
        }

import math

ncols = 2
nrows = math.ceil(len(nsteps_per_leg) / ncols)

fig = plt.figure(figsize=(14, 6 * nrows))
outer = fig.add_gridspec(
    nrows, ncols,
    hspace=0.30,
    wspace=0.18,
)

styles = {
    "compression": ("tab:blue", "Mean compression"),
    "expansion": ("tab:orange", "Mean expansion"),
}

for panel_index, nsteps in enumerate(nsteps_per_leg):
    row, column = divmod(panel_index, ncols)

    inner = outer[row, column].subgridspec(
        2, 1,
        height_ratios=[3, 1],
        hspace=0.05,
    )
    ax = fig.add_subplot(inner[0])
    ax_residual = fig.add_subplot(inner[1], sharex=ax)

    for direction in ("compression", "expansion"):
        curve = averaged_curves[nsteps][direction]
        color, label = styles[direction]

        ax.plot(
            curve["density"],
            curve["pressure"],
            color=color,
            lw=2.5,
            label=label,
        )
        ax.fill_between(
            curve["density"],
            curve["pressure"] - curve["std"],
            curve["pressure"] + curve["std"],
            color=color,
            alpha=0.12,
        )

    compression = averaged_curves[nsteps]["compression"]
    expansion = averaged_curves[nsteps]["expansion"]

    rho_min = max(
        compression["density"].min(),
        expansion["density"].min(),
    )
    rho_max = min(
        compression["density"].max(),
        expansion["density"].max(),
    )

    if rho_min >= rho_max:
        raise RuntimeError(
            f"Compression and expansion have no overlapping density range "
            f"for {nsteps:,} steps."
        )

    grid = np.linspace(rho_min, rho_max, density_grid_points)
    residual = (
        np.interp(
            grid,
            compression["density"],
            compression["pressure"],
        )
        - np.interp(
            grid,
            expansion["density"],
            expansion["pressure"],
        )
    )

    ax_residual.plot(grid, residual, color="black", lw=1.8)
    ax_residual.axhline(0, color="gray", ls="--", lw=1)

    ax.set_title(f"Nsteps per leg = {nsteps:,}")
    ax.set_ylabel("Mean pressure")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", labelbottom=False)

    ax_residual.set_xlabel(r"Density, $\rho=N/V$")
    ax_residual.set_ylabel(r"$P_{\mathrm{comp}}-P_{\mathrm{exp}}$")
    ax_residual.grid(alpha=0.3)

fig.suptitle(
    f"NVE constant-dV/dt triangle (single {resize_axis}-axis ramp)",
    fontsize=14,
    y=0.995,
)

plt.show()


In [ ]:
# --------------------------------------------------
# Pressure versus LJ time for every individual leg
# and the average at each Nsteps
# --------------------------------------------------

fig, (ax_comp, ax_exp) = plt.subplots(
    1,
    2,
    figsize=(16, 6),
    sharey=True,
)

direction_axes = {
    "compression": ax_comp,
    "expansion": ax_exp,
}

# One color for each Nsteps value.
colors = plt.cm.viridis(
    np.linspace(0.1, 0.9, len(nsteps_per_leg))
)

# Store the time-domain averages for later use.
time_averaged_curves = {}

for color, nsteps in zip(colors, nsteps_per_leg):
    time_averaged_curves[nsteps] = {}

    time_branches = {
        "compression": [],
        "expansion": [],
    }

    # Collect individual legs by direction.
    for run_id in sweep_runs[nsteps]:
        logs = open_run(run_id).logs_dataframe()

        lj_time = logs["this_lj_time"].to_numpy(dtype=float)
        density = logs["density"].to_numpy(dtype=float)
        pressure = logs["pressure"].to_numpy(dtype=float)

        direction = (
            "compression"
            if density[-1] > density[0]
            else "expansion"
        )

        # Sort by LJ time and remove duplicate time points.
        order = np.argsort(lj_time)
        lj_time = lj_time[order]
        pressure = pressure[order]

        lj_time, unique_indices = np.unique(
            lj_time,
            return_index=True,
        )
        pressure = pressure[unique_indices]

        time_branches[direction].append(
            {
                "run_id": run_id,
                "time": lj_time,
                "pressure": pressure,
            }
        )

    # --------------------------------------------------
    # Plot compression and expansion separately
    # --------------------------------------------------

    for direction in ("compression", "expansion"):
        ax = direction_axes[direction]
        runs = time_branches[direction]

        if len(runs) != number_of_cycles:
            raise RuntimeError(
                f"Expected {number_of_cycles} {direction} "
                f"legs for Nsteps={nsteps:,}, "
                f"but found {len(runs)}"
            )

        # Plot every individual cycle.
        for cycle_index, run_data in enumerate(runs):
            ax.plot(
                run_data["time"],
                run_data["pressure"],
                color=color,
                linewidth=1,
                alpha=0.35,
                label=(
                    f"{nsteps:,} steps: individual legs"
                    if cycle_index == 0
                    else None
                ),
            )

        # Use the time interval shared by all cycles.
        shared_time_min = max(
            run_data["time"].min()
            for run_data in runs
        )
        shared_time_max = min(
            run_data["time"].max()
            for run_data in runs
        )

        time_grid_points = max(
            len(run_data["time"])
            for run_data in runs
        )

        time_grid = np.linspace(
            shared_time_min,
            shared_time_max,
            time_grid_points,
        )

        pressure_on_grid = np.asarray([
            np.interp(
                time_grid,
                run_data["time"],
                run_data["pressure"],
            )
            for run_data in runs
        ])

        mean_pressure = pressure_on_grid.mean(axis=0)

        pressure_std = pressure_on_grid.std(
            axis=0,
            ddof=1 if len(runs) > 1 else 0,
        )

        smoothed_mean = (
            pd.Series(mean_pressure)
            .rolling(
                window=smoothing_window,
                center=True,
                min_periods=1,
            )
            .mean()
            .to_numpy()
        )

        time_averaged_curves[nsteps][direction] = {
            "lj_time": time_grid,
            "mean_pressure": mean_pressure,
            "smoothed_pressure": smoothed_mean,
            "pressure_std": pressure_std,
        }

        # Plot the average prominently.
        ax.plot(
            time_grid,
            smoothed_mean,
            color=color,
            linewidth=3,
            label=f"{nsteps:,} steps: mean",
        )

        # Cycle-to-cycle standard-deviation band.
        ax.fill_between(
            time_grid,
            smoothed_mean - pressure_std,
            smoothed_mean + pressure_std,
            color=color,
            alpha=0.10,
        )

# --------------------------------------------------
# Formatting
# --------------------------------------------------

ax_comp.set_xlim(-.1, 10)
ax_exp.set_xlim(-.1, 10)

ax_comp.set_title("Compression legs")
ax_exp.set_title("Expansion legs")

for ax in (ax_comp, ax_exp):
    ax.set_xlabel("LJ time within leg")
    ax.set_ylabel("Pressure")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)

fig.suptitle(
    f"NVE constant-dV/dt pressure versus LJ time "
    f"(single {resize_axis}-axis ramp)",
    fontsize=14,
)

plt.tight_layout()
plt.show()